### Step 2: Merge weather data into station data time series

This step enriches the bike-sharing time series data with meteorological variables such as hourly temperature (°C) and cumulative precipitation (mm) to enable enhanced feature engineering.

**Data Sources**

Free APIs (No Authentication Required)

Two primary options are available for retrieving local weather data without API credentials:

- Open-Meteo (https://open-meteo.com/): Provides global coverage using interpolated reanalysis data. Data is available for any latitude-longitude coordinate, making it ideal for locations with sparse weather station networks.

- Meteostat (https://github.com/meteostat/meteostat-python): Aggregates observations from nearby weather stations. Coverage depends on local station density but offers higher-quality observational data in well-instrumented regions.

APIs Requiring Authentication

For more comprehensive historical and observational data specific to Spain, consider obtaining an API key from AEMET:

- AEMET (https://opendata.aemet.es/centrodedescargas/obtencionAPIKey): Provides access to observational and historical climate data from the Spanish weather station network. Note that this service enforces rate limits on historical data requests.

Select one or more data sources and implement a data fetcher that integrates meteorological variables into your bike-sharing time series dataset. Consider the trade-offs between coverage, data quality, and API rate limitations when making your selection.

These could be valuable features for bike availability prediction:

- Wind speed discourages cycling
- Weather code categorical indicator (sunny=good, rain=bad)
- Apparent temperature better than raw temp for user behavior
                                                                                                                                           

In [1]:
#TODO
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

INFO: pip is looking at multiple versions of niquests to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of niquests to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [openmeteo-requests]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [requests-cache]


In [1]:
import openmeteo_requests
import pandas as pd
import numpy as np
import requests_cache
from retry_requests import retry
from datetime import datetime
import os
import glob

In [2]:


print("\n" + "="*70)
print("FETCHING BARCELONA WEATHER DATA FOR BIKE SHARING DATASET")
print("="*70)

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.weather_cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# Barcelona coordinates
latitude = 41.3851
longitude = 2.1734

# 1. FIND DATE RANGE FROM YOUR BIKE FILES
print("\n" + "="*70)
print("1. DETERMINING DATE RANGE FROM BIKE DATA")
print("="*70)

bike_files = [
    "bikes_2019.csv.gz",
    "bikes_2021.csv.gz", 
    "bikes_2022.csv.gz",
    "bikes_2023.csv.gz",
    "bikes_2024.csv.gz"
]

# Check which files actually exist
existing_files = [f for f in bike_files if os.path.exists(f)]
print(f"Found {len(existing_files)} bike data files:")
for f in existing_files:
    size_mb = os.path.getsize(f) / (1024**2)
    print(f"  - {f} ({size_mb:.0f} MB)")

if not existing_files:
    print("❌ No bike data files found! Please check the file names.")
    exit()

# Find min and max dates across all files
min_date = None
max_date = None

for file in existing_files:
    print(f"\nReading {file} to check date range...")
    # Read just the date column (much faster)
    df_sample = pd.read_csv(file, compression='gzip', usecols=['date'], nrows=1)
    
    # Check if 'date' column exists
    if 'date' not in df_sample.columns:
        print(f"  ⚠️ No 'date' column found in {file}, checking for alternatives...")
        # Try to read first row to see columns
        df_cols = pd.read_csv(file, compression='gzip', nrows=1)
        print(f"  Columns in file: {list(df_cols.columns)}")
        
        # Look for any date-like column
        date_cols = [col for col in df_cols.columns if 'date' in col.lower() or 'time' in col.lower()]
        if date_cols:
            date_col = date_cols[0]
            print(f"  Using '{date_col}' as date column")
            df_date = pd.read_csv(file, compression='gzip', usecols=[date_col])
            df_date.columns = ['date']  # Rename to 'date'
        else:
            print(f"  ❌ Cannot find date column in {file}")
            continue
    else:
        # Read just the date column
        df_date = pd.read_csv(file, compression='gzip', usecols=['date'])
    
    # Convert to datetime
    df_date['date'] = pd.to_datetime(df_date['date'])
    
    file_min = df_date['date'].min()
    file_max = df_date['date'].max()
    print(f"  Date range: {file_min.date()} to {file_max.date()}")
    
    if min_date is None or file_min < min_date:
        min_date = file_min
    if max_date is None or file_max > max_date:
        max_date = file_max

if min_date is None or max_date is None:
    print("❌ Could not determine date range from files")
    exit()

print(f"\n✅ Overall date range across all files:")
print(f"   Start: {min_date.date()}")
print(f"   End: {max_date.date()}")
print(f"   Total days: {(max_date - min_date).days + 1} days")




FETCHING BARCELONA WEATHER DATA FOR BIKE SHARING DATASET

1. DETERMINING DATE RANGE FROM BIKE DATA
Found 5 bike data files:
  - bikes_2019.csv.gz (56 MB)
  - bikes_2021.csv.gz (101 MB)
  - bikes_2022.csv.gz (103 MB)
  - bikes_2023.csv.gz (97 MB)
  - bikes_2024.csv.gz (27 MB)

Reading bikes_2019.csv.gz to check date range...
  Date range: 2019-03-24 to 2019-12-31

Reading bikes_2021.csv.gz to check date range...
  Date range: 2021-01-01 to 2021-12-31

Reading bikes_2022.csv.gz to check date range...
  Date range: 2022-01-01 to 2022-12-31

Reading bikes_2023.csv.gz to check date range...
  Date range: 2023-01-01 to 2023-12-31

Reading bikes_2024.csv.gz to check date range...
  Date range: 2024-01-01 to 2024-03-31

✅ Overall date range across all files:
   Start: 2019-03-24
   End: 2024-03-31
   Total days: 1835 days


In [3]:
# 2. FETCH WEATHER DATA
print("\n" + "="*70)
print("2. FETCHING WEATHER DATA FROM OPEN-METEO")
print("="*70)

# Format dates for API
start_date = min_date.strftime('%Y-%m-%d')
end_date = max_date.strftime('%Y-%m-%d')

print(f"\nFetching weather for Barcelona ({latitude}, {longitude})")
print(f"Period: {start_date} to {end_date}")

# Weather variables - keeping exactly what we want
hourly_variables = [
    "temperature_2m",         # Base temperature
    "apparent_temperature",   # "Feels like" temperature (better for human behavior)
    "precipitation",          # Rain (mm)
    "windspeed_10m",          # Wind speed (km/h)
    "weathercode"             # WMO weather code (for categorical conditions)
]

# API parameters
params = {
    "latitude": latitude,
    "longitude": longitude,
    "start_date": start_date,
    "end_date": end_date,
    "hourly": hourly_variables,
    "timezone": "Europe/Madrid"
}

print("\nMaking API request...")
responses = openmeteo.weather_api("https://archive-api.open-meteo.com/v1/archive", params=params)

# Process the response
response = responses[0]

print(f"\nLocation: {response.Latitude():.4f}°N, {response.Longitude():.4f}°E")
print(f"Elevation: {response.Elevation()} m asl")

# Process hourly data
hourly = response.Hourly()
hourly_data = {}

# Create datetime index
hourly_data["date"] = pd.date_range(
    start=pd.to_datetime(hourly.Time(), unit="s"),
    end=pd.to_datetime(hourly.TimeEnd(), unit="s"),
    freq=pd.Timedelta(seconds=hourly.Interval()),
    inclusive="left"
)

# Extract each weather variable
for i, var_name in enumerate(hourly_variables):
    var_data = hourly.Variables(i)
    if var_data is not None:
        hourly_data[var_name] = var_data.ValuesAsNumpy()
        print(f"  ✓ Loaded {var_name}: {len(hourly_data[var_name])} values")
    else:
        print(f"  ✗ Missing {var_name}")
        hourly_data[var_name] = np.nan

# Create weather dataframe
weather_df = pd.DataFrame(hourly_data)

print(f"\n✅ Weather data fetched successfully!")
print(f"   Shape: {weather_df.shape}")
print(f"   Date range: {weather_df['date'].min()} to {weather_df['date'].max()}")
print(f"   Total hours: {len(weather_df):,}")



2. FETCHING WEATHER DATA FROM OPEN-METEO

Fetching weather for Barcelona (41.3851, 2.1734)
Period: 2019-03-24 to 2024-03-31

Making API request...

Location: 41.4411°N, 2.2014°E
Elevation: 28.0 m asl
  ✓ Loaded temperature_2m: 44040 values
  ✓ Loaded apparent_temperature: 44040 values
  ✓ Loaded precipitation: 44040 values
  ✓ Loaded windspeed_10m: 44040 values
  ✓ Loaded weathercode: 44040 values

✅ Weather data fetched successfully!
   Shape: (44040, 6)
   Date range: 2019-03-23 23:00:00 to 2024-03-31 22:00:00
   Total hours: 44,040


In [4]:
# 3. ADD WEATHER CODE CATEGORIES
print("\n" + "="*70)
print("3. ADDING WEATHER CODE CATEGORIES")
print("="*70)

# Weather condition categories based on WMO codes
def get_weather_category(code):
    """Convert WMO weather code to categorical labels"""
    if pd.isna(code):
        return "unknown"
    code = int(code)
    if code == 0:
        return "clear"
    elif code in [1, 2, 3]:
        return "partly_cloudy"
    elif code in [45, 48]:
        return "foggy"
    elif code in [51, 53, 55, 56, 57]:
        return "drizzle"
    elif code in [61, 63, 65, 66, 67, 80, 81, 82]:
        return "rain"
    elif code in [71, 73, 75, 77, 85, 86]:
        return "snow"
    elif code in [95, 96, 99]:
        return "thunderstorm"
    else:
        return "other"

# Create categorical feature
weather_df['weather_category'] = weather_df['weathercode'].apply(get_weather_category)

# Also create a severity score from weathercode (0-3 scale)
def get_weather_severity(code):
    """Convert WMO code to severity score (0 = best, 3 = worst)"""
    if pd.isna(code):
        return 0
    code = int(code)
    if code in [0, 1, 2, 3]:  # Clear to partly cloudy
        return 0
    elif code in [45, 48]:  # Fog
        return 1
    elif code in [51, 53, 55, 56, 57]:  # Drizzle
        return 1
    elif code in [61, 63, 65, 80, 81, 82]:  # Rain
        return 2
    elif code in [66, 67, 71, 73, 75, 77, 85, 86]:  # Freezing rain or snow
        return 3
    elif code in [95, 96, 99]:  # Thunderstorm
        return 3
    else:
        return 0

weather_df['weather_severity'] = weather_df['weathercode'].apply(get_weather_severity)

# Simple binary indicators from the raw data
weather_df['is_raining'] = weather_df['precipitation'] > 0.1
weather_df['is_windy'] = weather_df['windspeed_10m'] > 30

# Add time features for merging
weather_df['year'] = weather_df['date'].dt.year
weather_df['month'] = weather_df['date'].dt.month
weather_df['day'] = weather_df['date'].dt.day
weather_df['hour'] = weather_df['date'].dt.hour

# Show summary
print("\nWeather category distribution:")
cat_counts = weather_df['weather_category'].value_counts()
for cat, count in cat_counts.items():
    print(f"  {cat}: {count:,} hours ({count/len(weather_df)*100:.1f}%)")

print("\nWeather severity distribution:")
sev_counts = weather_df['weather_severity'].value_counts().sort_index()
for sev, count in sev_counts.items():
    severity_names = ["Good", "Mild", "Bad", "Severe"]
    print(f"  {severity_names[sev]}: {count:,} hours ({count/len(weather_df)*100:.1f}%)")

print(f"\nTemperature range: {weather_df['temperature_2m'].min():.1f}°C to {weather_df['temperature_2m'].max():.1f}°C")
print(f"Apparent temperature range: {weather_df['apparent_temperature'].min():.1f}°C to {weather_df['apparent_temperature'].max():.1f}°C")
print(f"Rainy hours: {weather_df['is_raining'].sum():,} ({weather_df['is_raining'].mean()*100:.1f}%)")
print(f"Windy hours: {weather_df['is_windy'].sum():,} ({weather_df['is_windy'].mean()*100:.1f}%)")



3. ADDING WEATHER CODE CATEGORIES

Weather category distribution:
  partly_cloudy: 23,283 hours (52.9%)
  clear: 16,707 hours (37.9%)
  drizzle: 3,396 hours (7.7%)
  rain: 646 hours (1.5%)
  snow: 8 hours (0.0%)

Weather severity distribution:
  Good: 39,990 hours (90.8%)
  Mild: 3,396 hours (7.7%)
  Bad: 646 hours (1.5%)
  Severe: 8 hours (0.0%)

Temperature range: -0.7°C to 36.3°C
Apparent temperature range: -4.9°C to 38.4°C
Rainy hours: 2,699 (6.1%)
Windy hours: 316 (0.7%)


In [5]:
# 4. SAVE WEATHER DATA
print("\n" + "="*70)
print("4. SAVING WEATHER DATA")
print("="*70)

weather_filename = "barcelona_weather_2019_2024.csv.gz"
weather_df.to_csv(weather_filename, index=False, compression='gzip')

file_size_mb = os.path.getsize(weather_filename) / (1024**2)
print(f"✓ Saved to: {weather_filename}")
print(f"  File size: {file_size_mb:.1f} MB")
print(f"  Records: {len(weather_df):,}")
print(f"  Columns: {list(weather_df.columns)}")

# 5. SHOW SAMPLE
print("\n" + "="*70)
print("5. SAMPLE WEATHER DATA")
print("="*70)

sample_cols = ['date', 'temperature_2m', 'apparent_temperature', 'precipitation', 
               'windspeed_10m', 'weathercode', 'weather_category', 'weather_severity', 'is_raining']
print(weather_df[sample_cols].head(10).to_string())


4. SAVING WEATHER DATA
✓ Saved to: barcelona_weather_2019_2024.csv.gz
  File size: 0.7 MB
  Records: 44,040
  Columns: ['date', 'temperature_2m', 'apparent_temperature', 'precipitation', 'windspeed_10m', 'weathercode', 'weather_category', 'weather_severity', 'is_raining', 'is_windy', 'year', 'month', 'day', 'hour']

5. SAMPLE WEATHER DATA
                 date  temperature_2m  apparent_temperature  precipitation  windspeed_10m  weathercode weather_category  weather_severity  is_raining
0 2019-03-23 23:00:00       11.056000              7.733118            0.0      10.009036          0.0            clear                 0       False
1 2019-03-24 00:00:00       11.806000              8.254985            0.0      10.883676          0.0            clear                 0       False
2 2019-03-24 01:00:00       11.156000              7.309334            0.0      12.031756          1.0    partly_cloudy                 0       False
3 2019-03-24 02:00:00       11.606000              7.64851

In [6]:
import pandas as pd
import os

print("\n" + "="*70)
print("MERGING WEATHER DATA WITH ALL BIKE FILES")
print("="*70)

# Load weather data
print("Loading weather data...")
weather = pd.read_csv('barcelona_weather_2019_2024.csv.gz', compression='gzip')
weather['date'] = pd.to_datetime(weather['date'])
print(f"✓ Weather data loaded: {len(weather):,} records")

# List all your bike files
bike_files = [
    "bikes_2019.csv.gz",
    "bikes_2021.csv.gz", 
    "bikes_2022.csv.gz",
    "bikes_2023.csv.gz",
    "bikes_2024.csv.gz"
]

# Merge each year
for bike_file in bike_files:
    if os.path.exists(bike_file):
        print(f"\nProcessing {bike_file}...")
        
        # Load bike data
        bike_df = pd.read_csv(bike_file, compression='gzip')
        bike_df['date'] = pd.to_datetime(bike_df['date'])
        print(f"  Bike data: {len(bike_df):,} records")
        
        # Merge with weather
        merged_df = pd.merge(bike_df, weather, on='date', how='left')
        print(f"  Merged: {len(merged_df):,} records")
        
        # Check if any rows didn't get weather data
        missing_weather = merged_df['temperature_2m'].isnull().sum()
        if missing_weather > 0:
            print(f"  ⚠️ {missing_weather} rows missing weather data")
        
        # Save merged file
        output_file = bike_file.replace('.csv.gz', '_with_weather.csv.gz')
        merged_df.to_csv(output_file, index=False, compression='gzip')
        print(f"  ✓ Saved: {output_file}")
        
        # Show sample
        print(f"\n  Sample from {bike_file}:")
        print(merged_df[['date', 'bikes_available', 'temperature_2m', 'precipitation', 'weather_category']].head(3))
    else:
        print(f"\n⚠️ File not found: {bike_file}")

print("\n" + "="*70)
print("✅ MERGING COMPLETE")
print("="*70)
print("\nYou now have files like:")
print("  - bikes_2019_with_weather.csv.gz")
print("  - bikes_2021_with_weather.csv.gz")
print("  - bikes_2022_with_weather.csv.gz")
print("  - bikes_2023_with_weather.csv.gz")
print("  - bikes_2024_with_weather.csv.gz")


MERGING WEATHER DATA WITH ALL BIKE FILES
Loading weather data...
✓ Weather data loaded: 44,040 records

Processing bikes_2019.csv.gz...
  Bike data: 2,627,727 records
  Merged: 2,627,727 records
  ✓ Saved: bikes_2019_with_weather.csv.gz

  Sample from bikes_2019.csv.gz:
                 date  bikes_available  temperature_2m  precipitation  \
0 2019-03-24 13:00:00                0          19.556            0.0   
1 2019-03-24 13:00:00                0          19.556            0.0   
2 2019-03-28 17:00:00               20          14.606            0.0   

  weather_category  
0            clear  
1            clear  
2            clear  

Processing bikes_2021.csv.gz...
  Bike data: 4,413,152 records
  Merged: 4,413,152 records
  ✓ Saved: bikes_2021_with_weather.csv.gz

  Sample from bikes_2021.csv.gz:
        date  bikes_available  temperature_2m  precipitation weather_category
0 2021-01-01               27           6.406            0.0    partly_cloudy
1 2021-01-01               